In [1]:
# --- Dataerai provenance (optional) -----------------------------------------
# Traces this notebook run - cell source, outputs, logs, transfers, and
# environment - to the Dataerai platform when the SDK and daemon are
# available. Without them the notebook runs exactly as before.
try:
    %load_ext dataerai.magics
    %dataerai --trace --notebook graphene_bilayer_stem_simulations.ipynb abTEM / notebook runs / articles
except Exception as _dataerai_error:
    print(f"Dataerai tracing not active: {_dataerai_error}")

Signed in as demo@dataerai.com
Dataerai destination: abTEM / notebook runs / articles
Tracing notebook run 63e5f4cb-4aa3-45f6-909e-a2d3b86765fe. Cell source, outputs, logs, transfers, and environment details will be uploaded when %dataerai --finish runs.


# Simulations of 4D-STEM dataset of graphene bilayer

This is a notebook accompanying "Interferometric 4D-STEM for Lattice Distortion and Interlayer Spacing Measurements of Bilayer and Trilayer 2D Materials" ([doi:10.1002/smll.202100388](https://doi.org/10.1002/smll.202100388)) demonstrating the simulation of ADF images and 4D-STEM data of twisted bilayer graphene.

> **Note (2026):** this notebook was modernized from the abTEM 1.0-beta API it
> was published with to current abTEM 1.x: the scattering-matrix workflow now
> uses `SMatrix.scan(..., ctf=...)`, measurements are saved as Zarr instead of
> the removed HDF5 writer, and virtual detectors/center-of-mass are methods on
> `DiffractionPatterns`. Physical parameters are unchanged; the original code
> is preserved in the git history.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read

import abtem
from abtem import (
    CTF,
    AnnularDetector,
    GridScan,
    PixelatedDetector,
    Potential,
    Probe,
    SMatrix,
    dataerai,
    from_zarr,
    orthogonalize_cell,
    show_atoms,
)

abtem.config.set({"diagnostics.progress_bar": False});

Running the notebook on a fast CPU should take a few minutes. If you have a GPU available for CuPy, you can also switch the device below for even faster simulation.

In [3]:
device = 'cpu'

A sheet of 3.15 deg. twisted bilayer graphene is imported, the structure have been relaxed using density functional theory with the GPAW code (please see article).

In [4]:
atoms = read('gra_3.15deg_bilayer_relaxed.cif')

atoms = orthogonalize_cell(atoms)

atoms *= (2, 1, 1)

atoms.center(axis=2, vacuum=5)

gridscan = GridScan((0, 0), (atoms.cell[0, 0] / 2, atoms.cell[1, 1]), sampling=0.2)

show_atoms(atoms);

We then create an IAM potential based on the parameterization by Lobato and van Dyck (abTEM default) with 1024 gridpoints.

In [5]:
potential = Potential(atoms, gpts=1024, device=device).build()

In [6]:
# Open a Dataerai provenance run. A %dataerai trace is active (top
# cell), so the artifacts captured below become products of the
# notebook execution log and are linked into its lineage.
exp = dataerai.start_run(
    name="graphene bilayer 4D-STEM",
    collection="abTEM / notebook runs / articles",
)
exp.capture_structure(atoms)
exp.capture_potential(potential)

'potential-potentialarray'

# ADF

As an initial test, we can simulate an annular dark field image of the specimen (note that this was not included in the published article).

In [7]:
S = SMatrix(
    semiangle_cutoff=31,  # mrad
    energy=60e3,  # eV
    potential=potential,
    interpolation=8,
    device=device,
    store_on_host=True,
)

adf_probe = Probe(energy=60e3, semiangle_cutoff=31)
adf_scan = GridScan(
    (0, 0),
    (atoms.cell[0, 0] / 2, atoms.cell[1, 1]),
    sampling=0.9 * adf_probe.aperture.nyquist_sampling,
)

print('Scan sampling:', adf_scan.sampling, 'Å')
print('Real space sampling:', S.sampling, 'Å')

Scan sampling: (0.351875, 0.3529935220493314) Å
Real space sampling: (0.08796875, 0.07618317223916234) Å


In [8]:
exp.capture_illumination(adf_probe)
exp.capture_scan(adf_scan)

'scan-gridscan'

In [9]:
detector = AnnularDetector(inner=50, outer=150)
exp.capture_detector(detector)
measurement = S.scan(scan=adf_scan, detectors=detector).compute()
exp.capture_measurement(measurement, name="adf")

'measurement-adf'

In [10]:
measurement.tile((2, 1)).interpolate(0.2).show(figsize=(7, 7));

# 4D-STEM

The 4D-STEM dataset uses a small 8 mrad probe defocused by 1000 Å. In modern abTEM the defocus is applied through a contrast transfer function at scattering-matrix reduction time, and the diffraction patterns are downsampled to 60 mrad to save disk space and memory.

In [11]:
S_4d = SMatrix(
    semiangle_cutoff=8,  # mrad
    energy=60e3,  # eV
    potential=potential,
    interpolation=1,  # no interpolation, we want good Fourier space sampling
    downsample=60,  # downsample the scattering matrix to 60 mrad
    device=device,
    store_on_host=True,
)

ctf_4d = CTF(energy=60e3, semiangle_cutoff=8, defocus=1000)

probe_4d = Probe(energy=60e3, semiangle_cutoff=8, defocus=1000)
scan_4d = GridScan(
    (0, 0),
    (atoms.cell[0, 0] / 2, atoms.cell[1, 1]),
    sampling=0.9 * probe_4d.aperture.nyquist_sampling,
)

print('Scan sampling:', scan_4d.sampling, 'Å')

Scan sampling: (1.3648484848484848, 1.3450270409121075) Å


In [12]:
detector_4d = PixelatedDetector(max_angle=60)
exp.capture_illumination(probe_4d)
exp.capture_scan(scan_4d)
exp.capture_detector(detector_4d)
measurement_4d = S_4d.scan(scan=scan_4d, detectors=detector_4d, ctf=ctf_4d)
measurement_4d.to_zarr('grabi_4dstem_data.zarr.zip', overwrite=True)

'grabi_4dstem_data.zarr.zip'

In [13]:
measurement = from_zarr('grabi_4dstem_data.zarr.zip').compute()
measurement[0, 0].show();

We can bandlimit the patterns from the inside to eliminate the central disk for display.

In [14]:
measurement.bandlimit(inner=10)[0, 0].show(cmap='magma');

Before calculating the center of mass we bandlimit from the outside to select the maximum integrated angle.

In [15]:
com = measurement.bandlimit(0, 10).center_of_mass()
com_magnitude = (
    com.abs()
    if hasattr(com, 'abs')
    else abtem.Images(np.abs(com.array), sampling=com.sampling, metadata=com.metadata)
)

In [16]:
com_magnitude.tile((2, 1)).interpolate(0.1).show(figsize=(8, 6), cbar=True);

In [17]:
# Finalize the provenance run: uploads every captured artifact as a
# product of the notebook trace and records its derived->origin lineage.
dataerai.finish_run()

In [18]:
# --- Dataerai provenance: publish the execution trace, if one is active -----
try:
    %dataerai --finish
except Exception as _dataerai_error:
    print(f"Dataerai trace not published: {_dataerai_error}")

Notebook trace will publish after this cell finishes.


Published notebook execution trace 63e5f4cb-4aa3-45f6-909e-a2d3b86765fe (17 cells, 9 products).
